# Ingest File Autoloader

**Author:** Databricks Ingestion Team  
**Version:** 1.0  
**Last Updated:** 2026-03-31

This notebook ingests files using Databricks Autoloader. **Default: Incremental Load** (only new data is ingested). Set the load type widget to 'full' for a full reload.

---

**Parameters:**
- `input_path`: Path to the source files
- `output_table`: Destination table name
- `load_type`: 'incremental' (default) or 'full'

---

**Instructions:**
1. Set the input path, output table, and load type using the widgets below.
2. Run all cells to ingest data as per the selected load type.


In [ ]:
# Databricks widgets for parameterization
import pyspark.sql.functions as F
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("ingest_file_autoloader")

dbutils.widgets.text("input_path", "", "Input Path")
dbutils.widgets.text("output_table", "bronze_table", "Output Table")
dbutils.widgets.dropdown("load_type", "incremental", ["incremental", "full"], "Load Type")
input_path = dbutils.widgets.get("input_path")
output_table = dbutils.widgets.get("output_table")
load_type = dbutils.widgets.get("load_type")

try:
    df = (spark.read.format("cloudFiles")
          .option("cloudFiles.format", "json")
          .load(input_path))
    logger.info(f"Loaded {df.count()} records from {input_path}")
    if load_type == "full":
        df.write.format("delta").mode("overwrite").saveAsTable(output_table)
        logger.info(f"Full load: Overwrote {output_table}")
    else:
        df.write.format("delta").mode("append").saveAsTable(output_table)
        logger.info(f"Incremental load: Appended to {output_table}")
    print(f"Ingestion ({load_type}) successful.")
except Exception as e:
    logger.error(f"Error in ingestion: {e}")
    print(f"Error: {e}")


## Tracking Path Helpers
Functions to build schema and checkpoint tracking paths.

In [ ]:
def _build_tracking_paths(global_config: dict, source: dict) -> tuple[str, str]:
    dbx_cfg = global_config.get('databricks', {})
    checkpoint_root = dbx_cfg.get('checkpoint_root')
    schema_root = dbx_cfg.get('schema_tracking_root')
    tenant = source.get('tenant')
    product = source.get('product_name')
    source_system = source.get('source_system')
    source_entity = source.get('source_entity')
    required = {
        'checkpoint_root': checkpoint_root,
        'schema_tracking_root': schema_root,
        'tenant': tenant,
        'product_name': product,
        'source_system': source_system,
        'source_entity': source_entity,
    }
    missing = [k for k, v in required.items() if not v]
    if missing:
        raise ValueError(f'Missing tracking path configuration: {missing}')
    path_suffix = f

    checkpoint_location = f

    schema_location = f

    return schema_location, checkpoint_location

## Auto Loader Option Builder
Function to build Auto Loader options for a given source.

In [ ]:
def build_autoloader_options(source: dict, global_config: dict, source_options: dict | None = None) -> dict:
    # ...existing code...

## File Stream Ingestion
Function to start an Auto Loader structured streaming read.

In [ ]:
def ingest_file_stream(spark, source: dict, global_config: dict, source_options: dict | None = None):
    # ...existing code...

---

## Validation & Testing

Below, we validate the notebook logic with a simple assertion to ensure data was ingested. Add more tests as needed for your data.


In [ ]:
# Simple validation
try:
    df_test = spark.table(output_table)
    assert df_test.count() > 0, "No records found in output table."
    print("Validation passed: Data ingested.")
except Exception as e:
    print(f"Validation failed: {e}")


---

## Resource Cleanup

Unpersist DataFrames and clean up resources if needed to avoid memory issues in production workflows.

In [ ]:
# Resource cleanup
if 'df_test' in locals():
    df_test.unpersist(blocking=True)
    print("Resources cleaned up.")
